# <a id='toc1_'></a>[Laboratorio 1 - Exploración, preparación y regresión lineal](#toc0_)
*Presentado por:*
- Shaiel Jimenez - 202323846

- Juan Esteban Triviño - 202315338

*ISIS2611 - Aprendizaje de Máquina / Sección 5*

*Profesor: Rubén Manrique*

**Table of contents**<a id='toc0_'></a>    
- [Laboratorio 1 - Exploración, preparación y regresión lineal](#toc1_)    
- [Introducción](#toc2_)    
  - [Resumen de los pasos realizados](#toc2_1_)    
- [Exploración de los datos](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc2_'></a>[Introducción](#toc0_)

Este laboratorio aborda el caso del Instituto AlpesPlanck, que
registra variables meteorológicas en su estación de Jena, Alemania, y busca predecir
la temperatura máxima del día siguiente para anticipar condiciones que puedan derivar
en incendios forestales o en fenómenos como "El Niño" y "La Niña". A partir de estos
datos, en este trabajo se aplica el ciclo de machine learning para explorar y preparar
la información disponible, construir y comparar dos modelos de regresión lineal con
distintas estrategias de preparación, evaluar su desempeño mediante las métricas RMSE,
MAE y R², e identificar las variables meteorológicas que más aportan a la predicción,
con el fin de generar conocimiento útil para la toma de decisiones frente a este tipo
de fenómenos climáticos.

## <a id='toc2_1_'></a>[Resumen de los pasos realizados](#toc0_)

*(Esta sección se completa al finalizar el laboratorio, resumiendo brevemente las
decisiones tomadas en cada etapa del ciclo de ML.)*

- **Exploración de datos:** *[pendiente]*
- **Preparación de datos:** *[pendiente]*
- **Modelo 1 — estrategia de preparación:** *[pendiente]*
- **Modelo 2 — estrategia de preparación:** *[pendiente]*
- **Evaluación comparativa:** *[pendiente]*
- **Variables más importantes:** *[pendiente]*
- **Predicciones sobre datos de prueba:** *[pendiente]*

# <a id='toc3_'></a>[Exploración de los datos](#toc0_)

In [9]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

# Carga Inicial de datos

In [66]:
# cargar los datos empleando pandas
df = pd.read_csv('data/Datos Lab 1.csv')
data = df.copy()

df.shape 

(2576, 27)

## Revision y tratamiento de datos

In [67]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)


temp_max_manana      0.036879
viento_min           0.035326
estacion_anio        0.034161
mes                  0.034161
anio                 0.033385
humedad_max          0.033385
viento_media         0.033385
viento_max           0.031444
humedad_min          0.031056
viento_norte         0.031056
presion_desv         0.031056
rafaga_desv          0.030668
direccion_viento     0.030668
rafaga_media         0.030280
registros_del_dia    0.029891
presion_media        0.029115
presion_min          0.028727
humedad_media        0.028727
rafaga_max           0.028339
rafaga_min           0.027950
viento_desv          0.027950
fecha                0.027950
sector_viento        0.027562
dia_del_anio         0.026398
presion_max          0.024845
viento_este          0.024845
humedad_desv         0.023680
dtype: float64

In [68]:

# Si fecha es nulo se podria eliminar dado que no tiene sentido imputar una fecha, ademas de que es una columna que no aporta informacion para el analisis de los datos.

data = data.dropna(subset=['fecha'])

# Agrupar columnas numericas y categoricas

num_cols = data.select_dtypes(include=np.number).columns

cat_cols = data.select_dtypes(include='object').columns



num_imputer = SimpleImputer(strategy='median')  
data[num_cols] = num_imputer.fit_transform(data[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
data[cat_cols] = cat_imputer.fit_transform(data[cat_cols])
   












/tmp/ipykernel_410631/3636960105.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = data.select_dtypes(include='object').columns


In [69]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)

fecha                0.0
presion_media        0.0
presion_min          0.0
presion_max          0.0
presion_desv         0.0
humedad_media        0.0
humedad_min          0.0
humedad_max          0.0
humedad_desv         0.0
viento_media         0.0
viento_min           0.0
viento_max           0.0
viento_desv          0.0
rafaga_media         0.0
rafaga_min           0.0
rafaga_max           0.0
rafaga_desv          0.0
viento_norte         0.0
viento_este          0.0
direccion_viento     0.0
registros_del_dia    0.0
anio                 0.0
dia_del_anio         0.0
estacion_anio        0.0
mes                  0.0
sector_viento        0.0
temp_max_manana      0.0
dtype: float64

## Duplicados

In [70]:
n_duplicados = int(data.duplicated(keep=False).sum())
print(f"Número de registros duplicados: {n_duplicados}")

# Eliminar registros duplicados
data = data.drop_duplicates(keep='first')

n_duplicados_rev = int(data.duplicated(keep=False).sum())
print(f"Número de registros duplicados: {n_duplicados_rev}")

# ¿Hay fechas que se repiten?
dup_fechas = (data['fecha'].value_counts()
                            .loc[lambda s: s > 1]
                            .sort_values(ascending=False))
print(f"Fechas duplicadas: {len(dup_fechas)}")

for fecha, n in dup_fechas.items():
    print(f"Fecha={fecha} → {n} apariciones")



    

    


Número de registros duplicados: 8
Número de registros duplicados: 0
Fechas duplicadas: 14
Fecha=2009-03-05 → 2 apariciones
Fecha=2009-05-02 → 2 apariciones
Fecha=2009-12-08 → 2 apariciones
Fecha=2010-08-14 → 2 apariciones
Fecha=2010-10-25 → 2 apariciones
Fecha=2012-10-04 → 2 apariciones
Fecha=2012-10-06 → 2 apariciones
Fecha=2012-11-22 → 2 apariciones
Fecha=2013-01-02 → 2 apariciones
Fecha=2013-05-29 → 2 apariciones
Fecha=2015-01-07 → 2 apariciones
Fecha=2015-02-03 → 2 apariciones
Fecha=2015-05-11 → 2 apariciones
Fecha=2015-12-07 → 2 apariciones


In [71]:
data['fecha'] = pd.to_datetime(data['fecha'])

# Filas donde el año no coincide con la fecha
inconsistentes = data[data['anio'] != data['fecha'].dt.year]
print(inconsistentes.shape[0])
inconsistentes



69


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
25,2009-01-26,977.2310,975.000000,980.89,1.7438,96.563900,95.2000,97.5,0.5982,1.40990,...,1.0684,0.3670,18.9574,144.0,2012.0,26.0,verano,Sept,N,-1.04
30,2009-01-31,994.3756,989.950000,997.42,2.0522,0.884403,73.3000,94.4,5.4441,2.58660,...,1.9530,1.6026,39.3705,144.0,2012.0,31.0,Inverano,april,NE,14.36
121,2009-05-02,998.1189,995.990000,999.41,0.9205,77.500000,51.0900,99.6,18.9020,1.84440,...,-0.5643,0.4668,23.7431,144.0,2012.0,122.0,primavera,october,SO,20.89
132,2009-05-13,992.3840,989.590000,995.05,1.8406,59.670000,44.8800,75.7,10.2773,2.52420,...,1.3270,1.3113,44.6575,144.0,2012.0,133.0,primavera,Jul,NE,17.84
163,2009-06-13,993.6042,992.800000,994.35,0.4078,55.844900,30.5100,88.2,18.8033,6.93864,...,-0.2299,-1.5106,261.3481,144.0,2012.0,164.0,invierno,diciembre,O,23.83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2502,2015-11-08,999.8046,994.720000,1003.60,2.3725,74.341200,55.6800,90.7,9.0097,5.54148,...,-0.9200,-0.7359,218.6565,144.0,2012.0,312.0,otono,October,Suroeste,17.71
2536,2015-12-12,994.6981,998.714406,990.12,1.7716,80.644600,65.9200,92.2,7.5439,2.24340,...,-1.4604,-1.1378,217.9235,144.0,2012.0,346.0,winter,December,so,9.93
2551,2015-12-27,1000.5291,999.460000,1003.12,1.0983,77.068400,61.9100,87.6,7.6486,2.63670,...,-2.3958,-0.5269,192.4034,144.0,2012.0,361.0,invierno,December,S,9.72
2552,2015-12-28,1003.5725,1002.350000,1004.96,0.7294,87.109700,72.1000,97.7,7.5170,3.02724,...,-0.6644,-0.0853,187.3193,144.0,2012.0,362.0,verano,MARCH,S,9.16


In [72]:

data = data[data['anio'] == data['fecha'].dt.year]
print(data.shape)


(2431, 27)


In [73]:
dup_fechas_rev = (data['fecha'].value_counts()
                                .loc[lambda s: s > 1]
                                .sort_values(ascending=False))
print(f"Fechas duplicadas restantes: {len(dup_fechas_rev)}")

Fechas duplicadas restantes: 12


In [74]:
dup_table_rev = (
    data[data['fecha'].duplicated(keep=False)]
    .copy()
    .assign(repeticiones=data.groupby('fecha')['fecha'].transform('size'))
    .sort_values(['fecha'])
)
dup_table_rev





,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana,repeticiones
63,2009-03-05,959.860700,958.67000,962.54,0.8964,94.294400,86.0000,97.8,2.82960,1.29080,...,0.1826,9.3166,144.0,2009.0,64.0,primavera,July,N,5.370000,2
2574,2009-03-05,959.860700,958.67000,962.54,0.8964,68.529600,86.0000,97.8,2.82960,1.29080,...,0.1826,9.3166,144.0,2009.0,64.0,primavera,July,N,5.956009,2
2575,2009-12-08,983.849000,979.45000,991.40,3.7836,1.021309,0.8010,96.8,5.96410,4.53492,...,-0.3554,233.2095,144.0,2009.0,342.0,verano,Enero,SO,7.120000,2
341,2009-12-08,983.849000,979.45000,991.40,3.7836,0.877951,0.8010,96.8,5.96410,4.53492,...,-0.8314,233.2095,144.0,2009.0,342.0,verano,Enero,SO,7.120000,2
590,2010-08-14,990.482800,989.22000,991.29,0.4346,84.386400,65.1200,95.7,9.46810,1.70860,...,0.6511,39.5952,144.0,2010.0,226.0,verano,August,NE,19.500000,2
2566,2010-08-14,991.272748,989.22000,991.29,0.4346,84.386400,65.1200,95.7,9.46810,1.70860,...,0.6511,39.5952,144.0,2010.0,226.0,verano,August,NE,19.500000,2
662,2010-10-25,9928.772000,986.97000,998.17,3.4396,79.714900,62.7000,94.3,9.28340,5.98716,...,-1.2591,251.2583,144.0,2010.0,298.0,otono,October,O,7.340000,2
2562,2010-10-25,9928.772000,986.97000,998.17,3.4396,79.714900,49.7300,94.3,9.28340,5.98716,...,-1.2591,251.2583,144.0,2010.0,298.0,otono,October,O,7.340000,2
1372,2012-10-04,983.764000,977.38000,990.56,3.6130,73.460500,62.4100,88.9,7.29730,14.94468,...,-1.5376,203.8477,144.0,2012.0,278.0,otono,febrero,SO,63.050000,2
2557,2012-10-04,983.764000,977.38000,990.56,3.6130,73.460500,49.7300,88.9,10.20305,14.94468,...,-1.5376,203.8477,144.0,2012.0,278.0,otono,febrero,SO,63.050000,2


In [75]:
for fecha, grupo in data[data['fecha'].duplicated(keep=False)].groupby('fecha'):
    cols_distintas = grupo.loc[:, grupo.nunique().gt(1)]
    print(f"\nFecha: {fecha}")
    print(cols_distintas)


Fecha: 2009-03-05 00:00:00
      humedad_media  temp_max_manana
63          94.2944         5.370000
2574        68.5296         5.956009

Fecha: 2009-12-08 00:00:00
      humedad_media  viento_este
341        0.877951      -0.8314
2575       1.021309      -0.3554

Fecha: 2010-08-14 00:00:00
      presion_media
590      990.482800
2566     991.272748

Fecha: 2010-10-25 00:00:00
      humedad_min
662         62.70
2562        49.73

Fecha: 2012-10-04 00:00:00
      humedad_min  humedad_desv  viento_desv
1372        62.41       7.29730     242.9700
2557        49.73      10.20305       1.0573

Fecha: 2012-10-06 00:00:00
      presion_max  humedad_min  temp_max_manana
1374       991.84        49.73        13.820000
2567       988.75        57.30        14.798972

Fecha: 2012-11-22 00:00:00
      temp_max_manana
1421         7.190000
2573         7.647571

Fecha: 2013-01-02 00:00:00
      humedad_min
1462        49.73
2561        67.77

Fecha: 2013-05-29 00:00:00
      temp_max_manana
160

In [76]:
import pandas as pd

resumen = []
for fecha, grupo in data[data['fecha'].duplicated(keep=False)].groupby('fecha'):
    cols_dif = grupo.columns[grupo.nunique().gt(1)].tolist()
    resumen.append({'fecha': fecha, 'columnas_diferentes': cols_dif})

resumen_df = pd.DataFrame(resumen)
print(f"Fechas duplicadas encontradas: {len(resumen_df)}")

if len(resumen_df) == 0:
    print("No hay duplicados de fecha pendientes.")
else:
    rangos_plausibles = {
        'humedad_media': (0, 100), 'humedad_min': (0, 100), 'humedad_max': (0, 100),
        'humedad_desv': (0, 30), 'presion_media': (950, 1050), 'presion_min': (950, 1050),
        'presion_max': (950, 1050), 'viento_desv': (0, 15), 'viento_este': (-20, 20),
        'temp_max_manana': (-20, 45),
    }

    def clasificar_fecha(fecha, columnas):
        fecha_ts = pd.to_datetime(fecha)
        duplicados = data[data['fecha'] == fecha_ts]
        for col in columnas:
            if col not in rangos_plausibles:
                continue
            lo, hi = rangos_plausibles[col]
            if any(v < lo or v > hi for v in duplicados[col].values):
                return True
        return False

    resumen_df['es_corrupto'] = [
        clasificar_fecha(f, c) for f, c in zip(resumen_df['fecha'], resumen_df['columnas_diferentes'])
    ]
    print(resumen_df)

    fechas_corruptas = resumen_df.loc[resumen_df['es_corrupto'], 'fecha']
    fechas_ruido = resumen_df.loc[~resumen_df['es_corrupto'], 'fecha']

    print(f"\nShape antes: {data.shape}")
    data = data[~data['fecha'].isin(fechas_corruptas)]

    data_ruido = data[data['fecha'].isin(fechas_ruido)]
    promedios = data_ruido.groupby('fecha', as_index=False).mean(numeric_only=True)
    data = data[~data['fecha'].isin(fechas_ruido)]
    data = pd.concat([data, promedios], ignore_index=True)

    print(f"Shape después: {data.shape}")

dup_fechas_final = data['fecha'].value_counts().loc[lambda s: s > 1]
print(f"\nFechas duplicadas restantes: {len(dup_fechas_final)}")

Fechas duplicadas encontradas: 12
        fecha                          columnas_diferentes  es_corrupto
0  2009-03-05             [humedad_media, temp_max_manana]        False
1  2009-12-08                 [humedad_media, viento_este]        False
2  2010-08-14                              [presion_media]        False
3  2010-10-25                                [humedad_min]        False
4  2012-10-04     [humedad_min, humedad_desv, viento_desv]         True
5  2012-10-06  [presion_max, humedad_min, temp_max_manana]        False
6  2012-11-22                            [temp_max_manana]        False
7  2013-01-02                                [humedad_min]        False
8  2013-05-29                            [temp_max_manana]        False
9  2015-01-07                                [presion_min]        False
10 2015-05-11                              [humedad_media]        False
11 2015-12-07    [humedad_max, viento_desv, sector_viento]         True

Shape antes: (2431, 27)
Shape

In [78]:
data

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50000,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.77860,...,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.120000
1,2009-01-02,999.6006,997.93000,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.41950,...,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.820000
2,2009-01-03,998.5486,993.05000,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.25090,...,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.630000
3,2009-01-04,988.5107,985.12000,992.93,2.3223,89.417400,97.946704,93.2,4.4904,1.72040,...,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.440000
4,2009-01-05,990.4057,987.02000,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.80030,...,2.6275,0.2874,6.2418,144.0,2009.0,5.0,East,January,N,-10.880000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2412,2012-11-22,996.2830,1002.17446,992.92,1.4216,82.929500,60.890000,96.8,11.1910,1.56360,...,-1.4158,-0.3324,193.2139,144.0,2012.0,327.0,NaN,NaN,NaN,7.418785
2413,2013-01-02,9939.9180,985.61000,1001.18,4.9628,77.914400,58.750000,91.3,5.5425,9.23220,...,-1.9677,-1.4391,216.1814,144.0,2013.0,2.0,NaN,NaN,NaN,9.230000
2414,2013-05-29,974.2597,972.07000,979.05,2.1663,0.844688,0.700000,96.9,8.9043,7.09668,...,-1.5221,-0.6732,203.8588,144.0,2013.0,149.0,NaN,NaN,NaN,12.616204
2415,2015-01-07,999.4615,991.30500,1002.04,2.0521,81.148600,70.000000,92.0,6.0815,5.57748,...,-1.0755,-0.6137,209.7104,144.0,2015.0,7.0,NaN,NaN,NaN,44.978000
